# Missing Forest + Grafos + STN para PM2.5 en Bogotá

Este notebook propone una extensión del flujo actual para combinar:
- imputación con MissingForest
- construcción de un grafo de estaciones vecinas
- entrenamiento de un STN supervisado

La idea es documentar un pipeline reproducible para la siguiente etapa del proyecto.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from missingpy import MissForest

print('Librerías cargadas')

## 1) Carga de datos

En este punto se asume que ya existe la base de calidad del aire con variables meteorológicas y PM2.5.

In [ ]:
DATA_PATH = 'bogota_calidad_aire_raw.parquet'

try:
    df = pd.read_parquet(DATA_PATH)
    print('Filas:', len(df), '| columnas:', list(df.columns[:10]))
except Exception as e:
    print('No se encontró el parquet en la ruta esperada:', e)
    print('Ajusta DATA_PATH o usa la ruta completa del archivo.')
    df = pd.DataFrame()

## 2) Preprocesamiento y preparación

Se ordena por fecha y se construye una columna objetivo para PM2.5.

In [ ]:
if not df.empty:
    df = df.sort_values('fecha').copy() if 'fecha' in df.columns else df.copy()
    if 'pm2_5' in df.columns and 'pm2.5' not in df.columns:
        df = df.rename(columns={'pm2_5': 'pm2.5'})
    if 'pm2.5' not in df.columns:
        raise ValueError('No existe la variable objetivo pm2.5')
    df['fecha'] = pd.to_datetime(df['fecha']) if 'fecha' in df.columns else pd.RangeIndex(start=0, stop=len(df))
    print(df.head().to_string())

## 3) Imputación con MissingForest

MissingForest es útil para imputar datos faltantes preservando relaciones entre variables.

In [ ]:
if not df.empty:
    cols = [c for c in df.columns if c != 'fecha']
    X = df[cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    mf = MissForest(random_state=42, n_estimators=100, max_depth=20)
    X_imp = mf.fit_transform(X)
    df_imp = pd.DataFrame(X_imp, columns=cols, index=df.index)
    print('Imputación completada. Missing values:', df_imp.isna().sum().sum())
    print(df_imp.head())

## 4) Construcción de grafo de estaciones

Se asume que el dataframe puede ser expandido por estación. Si hay una columna `estacion`, se construye el grafo con vecinos por distancia o correlación.

In [ ]:
if not df.empty and 'estacion' in df.columns:
    estaciones = sorted(df['estacion'].unique())
    G = nx.Graph()
    for e in estaciones:
        G.add_node(e)
    # ejemplo simple: conectividad por correlación entre series de PM2.5
    series = []
    for e in estaciones:
        s = df[df['estacion'] == e]['pm2.5'].astype(float)
        series.append((e, s))
    for i in range(len(series)):
        for j in range(i + 1, len(series)):
            e1, s1 = series[i]
            e2, s2 = series[j]
            if s1.empty or s2.empty:
                continue
            corr = s1.corr(s2)
            if pd.notna(corr) and abs(corr) > 0.4:
                G.add_edge(e1, e2, weight=float(corr))
    print('Nodos:', G.number_of_nodes(), '| Aristas:', G.number_of_edges())
    nx.draw(G, with_labels=True, node_color='lightblue', font_size=9)
    plt.title('Grafo de estaciones basado en correlación')
    plt.show()
else:
    print('No hay columna estacion para construir grafo; aquí se queda como plantilla.')

## 5) Preparación para STN

Se sintetiza una versión mínima del input para el STN usando una ventana temporal y variables relevantes.

In [ ]:
if not df.empty:
    target_col = 'pm2.5'
    feat_cols = [c for c in df.columns if c not in ['fecha', 'estacion', target_col]]
    feat_cols = feat_cols[:10]  # solo para plantilla
    df_model = df[feat_cols + [target_col]].copy()
    df_model = df_model.replace([np.inf, -np.inf], np.nan)
    X = df_model[feat_cols].fillna(df_model[feat_cols].median())
    y = df_model[target_col].fillna(df_model[target_col].median())
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)
    model = RandomForestRegressor(random_state=42, n_estimators=200, max_depth=12)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f'RMSE={rmse:.3f} | MAE={mae:.3f} | R2={r2:.3f}')

## 6) Recomendación de próximo paso

Fase recomendada para la tesis o el análisis final:
1. Generar tabla con estaciones, correlaciones, vecinos y grafo.
2. Imputar con MissingForest por estación y luego combinar con variables vecinas.
3. Entrenar STN con ventanas temporales y comparar con persistencia y RF.
4. Reportar métricas: RMSE, MAE, R² y análisis de error por temporada.